# Parte 1 — Timed GNN (TimeGNN) per puzzle di "matto in n"

**Obiettivo.** Addestrare con la libreria **TimeGNN** (Wong & Damiani), *senza modificarla*, un modello che, data la storia delle mosse e la posizione, suggerisce la mossa successiva di un problema di matto in *n*. Si confrontano varianti **con** e **senza** informazione temporale (ablation).

**Come la libreria vede gli scacchi.** TimeGNN nasce per il process mining: ogni *case* è una sequenza di *eventi* con timestamp, trasformata in un grafo a catena (evento *j* → evento *j+1*); il modello predice l'evento successivo. La mappatura adottata:

| TimeGNN | Scacchi |
|---|---|
| case | un puzzle |
| event (classe da predire) | mossa UCI (1968 possibili + `EOS`) |
| time | istante della mossa (qui simulato: i puzzle non hanno clock) |
| feature del nodo | scacchiera **dopo** la mossa (64 case), tratto, scacco, pezzo mosso, tipo di mossa, mosse rimanenti, tempo speso |
| feature di sequenza | rating del puzzle, *n* |

**Varianti addestrate** (ognuna con più seed):

| ID | Modello TimeGNN | Tempo |
|---|---|---|
| M1_basic_notime | `DualGATModel` | no |
| M2_basic_time | `DualGATModel` (tempo come attributo d'arco) | sì |
| M3_decay_notime | `DualGATTimeAwareModel` (tempo azzerato) | no |
| M4_decay_time | `DualGATTimeAwareModel` (decadimento temporale dell'attenzione) | sì |

più due baseline senza apprendimento: **B0** mossa legale casuale, **B1** euristica a 1 semimossa (matto > scacco > cattura).

**Accorgimenti (tutti esterni alla libreria)**, motivati nelle celle dove vengono applicati:
1. `pandas<3` (con pandas 3 la libreria azzera i tempi);
2. un **processo separato per ogni variante** (in PyG ≥ 2.5 istanziare `GATConv` rompe `TimeAwareGATConv` nello stesso processo);
3. **self-loop** aggiunti ai grafi del modello time-decay;
4. **tempo causale** al posto del "tempo alla fine del case";
5. **vocabolario fisso** delle mosse e **split nostro** 80/10/10.

**Struttura dei file generati**
```
progetto/
├── 01_timegnn_training.ipynb   ← questo notebook
├── chess_tgnn.py               ← generato dalle celle %%writefile (Step 5, 7, 8)
├── train_variant.py            ← generato allo Step 9
├── train_config.json
├── data/                       ← dati (data_smoke/ in modalità smoke)
└── runs/                       ← un sotto-cartella per variante×seed
```

> **Prima esecuzione: lasciare `SMOKE = True`** (Step 2). Tutta la pipeline gira in pochi minuti su pochi dati; poi si passa a `SMOKE = False`.

## Step 1 — Ambiente

Da eseguire **una volta, nel terminale**, nella cartella del progetto (con `TimeGNN-main/` estratta accanto al notebook):

```bash
python3.12 -m venv .venv && source .venv/bin/activate
pip install --upgrade pip
pip install torch torch_geometric "pandas<3" python-chess zstandard pyarrow \
            scipy matplotlib requests ipykernel
pip install ./TimeGNN-main          # installazione normale: la libreria non viene toccata
python -m ipykernel install --user --name timegnn-chess
brew install stockfish              # fortemente raccomandato (vedi Step 8)
```

Note:
- **Non** usare l'extra `[pyg]` di TimeGNN: installerebbe `torch-scatter`/`torch-sparse`, che su macOS vanno compilati e non servono (PyG recente ne fa a meno).
- `pandas<3` è **obbligatorio**: pandas 3 crea timestamp in microsecondi, la libreria li divide come se fossero nanosecondi e i delta sotto ~1000 s diventano 0.
- Poi selezionare nel notebook il kernel `timegnn-chess`.

In [ ]:
import sys, platform, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import torch, torch_geometric, chess, sklearn
import timegnn

print(f"Python        {sys.version.split()[0]}  ({platform.platform()})")
print(f"pandas        {pd.__version__}")
print(f"torch         {torch.__version__}   MPS disponibile: {torch.backends.mps.is_available()}")
print(f"torch_geom.   {torch_geometric.__version__}")
print(f"python-chess  {chess.__version__}")
print(f"timegnn       {timegnn.__file__}")

assert int(pd.__version__.split(".")[0]) < 3, (
    "Serve pandas<3: con pandas 3 TimeGNN azzera i tempi (timestamp in µs invece di ns).")

## Step 2 — Configurazione

Tutte le scelte sperimentali in un posto solo.

**Dimensioni dei dati e RAM (24 GB).** Ogni nodo ha circa 820 feature (one-hot delle 64 case + altro). Con 12.000 puzzle per *n* (60.000 puzzle, circa 300.000 nodi) i grafi di training occupano circa 1–1,5 GB; il picco durante la costruzione è di qualche GB, perché la libreria crea matrici dense (lo mitighiamo trasformando a blocchi). C'è margine: si può salire a circa 25.000 per *n*, ma conviene prima misurare i tempi con la configurazione di default.

**Matti in 5: pochi.** Dopo i filtri Lichess ne offre solo qualche migliaio (circa 3.500), quindi per *n* = 5 il campione sarà più piccolo degli altri. Il notebook prende il minimo tra `PER_N` e il disponibile; va segnalato nel report.

**Early stopping.** `patience=6` e `min_epochs=10`. Il modello time-decay passa qualche epoca su un plateau, in cui predice sempre `EOS` (accuratezza circa 0,17), prima di iniziare a imparare. In una prova con 10.000 puzzle è uscito dal plateau alla 6ª–7ª epoca, e un early stopping troppo aggressivo lo avrebbe fermato prima.

**Iperparametri.** Valori nel range della traccia (lr 1e-3, batch 128). Due scelte da commentare:
- `num_layers=2` per percorso: con i self-loop, un nodo "vede" fino a circa 4 mosse precedenti. La maggior parte dell'informazione è comunque nella scacchiera del nodo stesso.
- `lambda_decay=1.0` invece del default della libreria (0,01). Con tempi scalati in [0, 1], λ = 0,01 dà un decadimento tra 0,99 e 1: il tempo sarebbe inerte e l'ablation non misurerebbe nulla. Con λ = 1 il decadimento varia tra 0,37 e 1.

In [ ]:
import os, json, shutil
from pathlib import Path

SMOKE = True                      # True = pipeline completa su pochi dati (minuti)

PROJECT_DIR = Path.cwd()
RAW_DIR  = PROJECT_DIR / "data" / "raw"            # file Lichess, condiviso smoke/full
DATA_DIR = PROJECT_DIR / ("data_smoke" if SMOKE else "data")
RUNS_DIR = PROJECT_DIR / ("runs_smoke" if SMOKE else "runs")
for d in (RAW_DIR, DATA_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED  = 42                        # seed dei DATI (campionamento, split, tempi simulati)
MAX_N = 5                         # matti in 1..5 per il training

# puzzle per valore di n (si prende il minimo tra questo e il disponibile)
PER_N = {n: 12_000 for n in range(1, MAX_N + 1)}
MAX_ROWS_SCAN   = None            # righe del CSV da leggere (None = tutte)
MIN_POPULARITY  = 50              # scarta puzzle poco apprezzati dagli utenti
MAX_RATING_DEV  = 90              # scarta puzzle con rating poco stabile
SPLIT_FRACS     = (0.8, 0.1, 0.1) # train / val / test (i classici held-out arrivano dopo)

TRAIN_CFG = dict(
    embedding_dims=64, gat_hidden_dim_event=64, gat_hidden_dim_embed=128,
    gat_hidden_dim_concat=128, num_heads=4, num_layers=2, dropout=0.1,
    use_batch_norm=True, activation="elu", lambda_decay=1.0,
    lr=1e-3, batch_size=128, epochs=40, patience=6, min_epochs=10, delta=1e-4,
    rollout_per_n=1000,           # puzzle di test per n risolti "per davvero" (il rollout è veloce)
    rollout_batch=512, data_seed=SEED,
)
SEEDS  = [0, 1, 2]
DEVICE = "auto"                   # "auto" | "mps" | "cpu"  (vedi Step 10)
STOCKFISH_PATH = shutil.which("stockfish") or ""

if SMOKE:
    PER_N = {n: 200 for n in range(1, MAX_N + 1)}
    MAX_ROWS_SCAN = 800_000
    TRAIN_CFG.update(epochs=2, rollout_per_n=15)
    SEEDS = [0]

CONFIG_PATH = PROJECT_DIR / ("train_config_smoke.json" if SMOKE else "train_config.json")
CONFIG_PATH.write_text(json.dumps(TRAIN_CFG, indent=2))
print("modalità:", "SMOKE" if SMOKE else "COMPLETA", "| dati:", DATA_DIR.name, "| run:", RUNS_DIR.name)
print("stockfish:", STOCKFISH_PATH or "non trovato (rollout in modalità stretta, vedi Step 8)")

## Step 3 — Scaricare i puzzle Lichess e tenere i matti in 1–5

Il database dei puzzle Lichess (circa 300 MB compressi, oltre 5 milioni di puzzle) contiene per ogni puzzle: `FEN`, `Moves` (UCI), `Rating`, `Popularity`, `Themes` e altro. I matti sono marcati con i temi `mateIn1` … `mateIn5`.

Il file viene letto **in streaming a blocchi** senza decomprimerlo su disco, così la RAM resta bassa. Si verifica anche che il numero di mosse sia coerente: un matto in *n* ha 2*n* mosse (la prima è dell'avversario).

In [ ]:
import io, re, time
import requests, zstandard

URL = "https://database.lichess.org/lichess_db_puzzle.csv.zst"
RAW_FILE = RAW_DIR / "lichess_db_puzzle.csv.zst"

if not RAW_FILE.exists():
    print("download in corso...")
    with requests.get(URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        total, done = int(r.headers.get("content-length", 0)), 0
        with open(RAW_FILE.with_suffix(".part"), "wb") as f:
            for block in r.iter_content(chunk_size=1 << 20):
                f.write(block); done += len(block)
                print(f"\r{done/1e6:7.1f} / {total/1e6:.1f} MB", end="")
    RAW_FILE.with_suffix(".part").rename(RAW_FILE)
    print("\nok")
print(RAW_FILE, f"{RAW_FILE.stat().st_size/1e6:.0f} MB")

In [ ]:
def load_mate_puzzles(path, max_n, max_rows=None, chunksize=250_000):
    # Legge il CSV .zst a blocchi e tiene solo i puzzle mateIn1..mateIn{max_n}.
    pattern = r"\bmateIn(\d+)\b"
    cols = ["PuzzleId", "FEN", "Moves", "Rating", "RatingDeviation",
            "Popularity", "NbPlays", "Themes", "GameUrl"]
    kept, seen = [], 0
    with open(path, "rb") as fh:
        text = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(fh), encoding="utf-8")
        for chunk in pd.read_csv(text, chunksize=chunksize, usecols=cols):
            seen += len(chunk)
            n = pd.to_numeric(chunk["Themes"].str.extract(pattern, expand=False), errors="coerce")
            chunk = chunk.assign(n=n)
            kept.append(chunk[chunk["n"].between(1, max_n)])
            print(f"\rrighe lette: {seen:,}", end="")
            if max_rows and seen >= max_rows:
                break
    print()
    df = pd.concat(kept, ignore_index=True)
    df["n"] = df["n"].astype(int)
    ok = df["Moves"].str.split().str.len() == 2 * df["n"]
    print(f"scartati per lunghezza incoerente: {(~ok).sum()}")
    return df[ok].reset_index(drop=True)

MATE_CACHE = DATA_DIR / "mate_puzzles.parquet"
t0 = time.time()
if MATE_CACHE.exists():
    mate = pd.read_parquet(MATE_CACHE)
else:
    mate = load_mate_puzzles(RAW_FILE, MAX_N, MAX_ROWS_SCAN)
    mate.to_parquet(MATE_CACHE)
print(f"{len(mate):,} puzzle di matto in {time.time()-t0:.0f}s")
mate.groupby("n").agg(puzzle=("PuzzleId", "size"), rating_medio=("Rating", "mean")).round(0)

## Step 4 — Pulizia, campionamento bilanciato e split

- **Deduplicazione** per posizione (`FEN`) e per partita di origine: più puzzle dalla stessa partita finirebbero in split diversi con posizioni quasi identiche (leakage).
- **Filtri di qualità**: popolarità minima e deviazione del rating massima.
- **Bilanciamento per n**: nei dati grezzi dominano mateIn1 e mateIn2. Campionando lo stesso numero di puzzle per *n* il modello non impara solo i matti corti, e le metriche per *n* sono confrontabili.
- **Split 80/10/10 stratificato per n**: *val* serve all'early stopping, *test* alla valutazione. Il test set della libreria non è usato, perché lo split interno delle recipe è deterministico per lunghezza.

In [ ]:
puz = mate.copy()
puz["game_id"] = puz["GameUrl"].str.extract(r"lichess\.org/(\w{8})", expand=False)
before = len(puz)
puz = puz.drop_duplicates("FEN").drop_duplicates("game_id")
puz = puz[(puz["Popularity"] >= MIN_POPULARITY) & (puz["RatingDeviation"] <= MAX_RATING_DEV)]
print(f"dopo dedup e filtri: {len(puz):,} / {before:,}")

parts = []
for n, g in puz.groupby("n"):
    k = min(PER_N[n], len(g))
    g = g.sample(k, random_state=SEED).reset_index(drop=True)
    cut1 = int(round(SPLIT_FRACS[0] * k)); cut2 = cut1 + int(round(SPLIT_FRACS[1] * k))
    g["split"] = np.where(g.index < cut1, "train", np.where(g.index < cut2, "val", "test"))
    parts.append(g)
puz = pd.concat(parts, ignore_index=True)
puz.to_parquet(DATA_DIR / "puzzles.parquet")

pd.crosstab(puz["n"], puz["split"], margins=True)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3.5))
for n, g in puz.groupby("n"):
    ax.hist(g["Rating"], bins=40, alpha=0.5, label=f"mateIn{n}")
ax.set_xlabel("rating del puzzle"); ax.set_ylabel("puzzle"); ax.legend()
ax.set_title("Distribuzione del rating per n (campione)")
plt.tight_layout(); plt.show()

## Step 5 — Modulo `chess_tgnn`, parte A: dagli scacchi all'event log

Il codice riusabile va in un modulo (`chess_tgnn.py`) perché lo userà anche lo script di training, che gira in un processo separato. Questa cella **crea** il file; le celle degli Step 7 e 8 lo **estendono** (`%%writefile -a`).

Scelte di rappresentazione:

1. **Forma canonica.** Nel CSV Lichess il FEN ha al tratto l'avversario, che gioca la prima mossa. Se chi deve dare matto è il Nero, specchiamo scacchiera e mosse: nel dataset il risolutore gioca sempre col Bianco. Il modello impara ogni pattern una volta sola invece di due. In inferenza la mossa si riporta all'orientamento originale.
2. **Un nodo = una mossa**, con le feature della scacchiera *dopo* quella mossa. TimeGNN associa al nodo *j* la label "mossa *j+1*", quindi il nodo contiene esattamente la posizione in cui quella mossa va trovata. L'ultimo nodo (dopo il matto) ha label `EOS`.
3. **Vocabolario fisso di 1968 mosse** (tutte quelle geometricamente possibili, promozioni incluse) più `EOS`.
4. **`moves_left`**: mosse ancora a disposizione del risolutore. È un'informazione legittima, nota anche in inferenza: all'LLM diremo "matto in *n*".
5. **Tempo simulato.** I puzzle non hanno clock. Come chiede la traccia, il tempo speso su ogni mossa è simulato dal rating: log-normale attorno a 4 s·e^{0,5·(rating−1500)/500}, e più breve per la mossa dell'avversario. La scala è fissa, `log1p(t)/log1p(600)`, quindi non c'è nessun fit sui dati e nessun leakage.

> ⚠️ **Da scrivere nel report:** il tempo simulato è una funzione del rating (più rumore) e il rating è già una feature. Sui puzzle ci aspettiamo quindi un effetto del tempo nullo o quasi. La vera verifica della domanda (2) richiede tempi reali (partite con clock, notebook successivo).

In [ ]:
%%writefile chess_tgnn.py
"""chess_tgnn - codice di supporto al progetto "Timed GNN per puzzle di matto in n".

QUESTO FILE E' GENERATO DAL NOTEBOOK 01_timegnn_training.ipynb (celle %%writefile).
Non modificarlo a mano: modifica la cella corrispondente e riesegui il notebook.

La libreria TimeGNN NON viene modificata: qui usiamo solo le sue classi pubbliche
(EventLogTransformer, DualGATModel, DualGATTimeAwareModel, train_epoch, ...).
Il modulo e' importato sia dal notebook sia da train_variant.py (processi separati).
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import chess
import numpy as np
import pandas as pd

# =============================================================================
# PARTE A - Dagli scacchi all'event log (Step 5 del notebook)
# =============================================================================

SQUARE_COLS: List[str] = [f"sq_{chess.square_name(s)}" for s in chess.SQUARES]
T_CAP_SECONDS = 600.0                      # tempo di riflessione massimo considerato
BASE_TS = pd.Timestamp("2020-01-01", tz="UTC")


def uci_universe() -> List[str]:
    """Tutte le mosse UCI geometricamente possibili (1968).

    Serve a fissare a priori il vocabolario delle classi: cosi' una mossa mai vista
    in training non fa fallire l'encoder su validation/test.
    """
    moves = set()
    for f in chess.SQUARES:
        for t in chess.SQUARES:
            if f == t:
                continue
            dfile = abs(chess.square_file(f) - chess.square_file(t))
            drank = abs(chess.square_rank(f) - chess.square_rank(t))
            if dfile == 0 or drank == 0 or dfile == drank or (dfile, drank) in ((1, 2), (2, 1)):
                moves.add(chess.square_name(f) + chess.square_name(t))
    for rank_from, rank_to in ((6, 7), (1, 0)):          # promozioni bianco / nero
        for file in range(8):
            f = chess.square(file, rank_from)
            for dfile in (-1, 0, 1):
                if 0 <= file + dfile < 8:
                    t = chess.square(file + dfile, rank_to)
                    for p in "qrbn":
                        moves.add(chess.square_name(f) + chess.square_name(t) + p)
    return sorted(moves)


UCI_UNIVERSE: List[str] = uci_universe()
# LabelEncoder ordina le classi: fittando anche il "caso vocabolario" (Parte B)
# le classi dell'encoder coincidono esattamente con questa lista.
CLASSES: List[str] = sorted(UCI_UNIVERSE + ["EOS"])
CLASS_INDEX: Dict[str, int] = {m: i for i, m in enumerate(CLASSES)}
EOS_INDEX: int = CLASS_INDEX["EOS"]


def mirror_move(move: chess.Move) -> chess.Move:
    """Specchia una mossa (a1<->a8), coerente con chess.Board.mirror()."""
    return chess.Move(chess.square_mirror(move.from_square),
                      chess.square_mirror(move.to_square),
                      promotion=move.promotion)


def canonicalize_puzzle(fen: str, moves_uci: Sequence[str]) -> Tuple[chess.Board, List[str], bool]:
    """Porta il puzzle in forma canonica: chi deve dare matto gioca col Bianco.

    Nel CSV Lichess il FEN ha al tratto l'AVVERSARIO: la prima mossa di `Moves`
    e' la sua, poi alternano risolutore/difensore. Se il risolutore e' il Nero
    specchiamo scacchiera e mosse: il modello impara i pattern una volta sola.
    """
    board = chess.Board(fen)
    solver_is_black = (not board.turn) == chess.BLACK
    moves = [chess.Move.from_uci(u) for u in moves_uci]
    if solver_is_black:
        board = board.mirror()
        moves = [mirror_move(m) for m in moves]
    return board, [m.uci() for m in moves], solver_is_black


def decanonicalize_uci(uci: str, mirrored: bool) -> str:
    """Riporta una mossa canonica nell'orientamento originale del puzzle."""
    return mirror_move(chess.Move.from_uci(uci)).uci() if mirrored else uci


def move_role(ply: int) -> str:
    """Ruolo della mossa in posizione `ply` della sequenza del puzzle."""
    if ply == 0:
        return "opp"                      # mossa dell'avversario che crea il puzzle
    return "solver" if ply % 2 == 1 else "def"


def move_flag(board_before: chess.Board, move: chess.Move) -> str:
    """Tipo di mossa (feature categorica). 'mate' viene assegnato dopo il push."""
    if board_before.is_castling(move):
        return "castle"
    if move.promotion is not None:
        return "promo"
    cap = board_before.is_capture(move)
    chk = board_before.gives_check(move)
    if cap and chk:
        return "capture_check"
    if chk:
        return "check"
    if cap:
        return "capture"
    return "quiet"


# --- Tempo simulato (i puzzle non hanno clock): vedi Step 5 del notebook ---------

def expected_think_time(rating: float, role: str) -> float:
    """Tempo di riflessione atteso in secondi: cresce col rating del puzzle."""
    base = 4.0 * math.exp(0.5 * (float(rating) - 1500.0) / 500.0)
    return base * (0.6 if role == "opp" else 1.0)


def simulate_think_time(rating: float, role: str, rng: np.random.Generator) -> float:
    """Campione log-normale attorno al tempo atteso (sigma=0.6), limitato a [0.5, 600] s."""
    mu = expected_think_time(rating, role)
    return float(np.clip(rng.lognormal(math.log(mu), 0.6), 0.5, T_CAP_SECONDS))


def scale_time(seconds: float) -> float:
    """Scala fissa (nessun fit sui dati -> nessun leakage): log1p(t)/log1p(600) in [0,1]."""
    return min(math.log1p(seconds) / math.log1p(T_CAP_SECONDS), 1.0)


def sequence_rows(case_id: str, start_board: chess.Board, moves: Sequence[str],
                  think_times: Sequence[float], rating: float, mate_n: int) -> List[dict]:
    """Converte UNA sequenza di mosse in righe di event log (una riga = un nodo).

    Il nodo j rappresenta la mossa j; le sue feature descrivono la scacchiera
    DOPO la mossa j, cioe' esattamente cio' che serve per predire la mossa j+1
    (la label che TimeGNN associa al nodo j).
    Usata sia per il training sia in inferenza (rollout): stessa featurizzazione.
    """
    board = start_board.copy(stack=False)
    rows, t_cum, solver_played = [], 0.0, 0
    for j, uci in enumerate(moves):
        mv = chess.Move.from_uci(uci)
        role = move_role(j)
        piece = board.piece_at(mv.from_square)
        mover = piece.symbol() if piece else "?"
        flag = move_flag(board, mv)
        board.push(mv)
        if board.is_checkmate():
            flag = "mate"
        if role == "solver":
            solver_played += 1
        t_cum += float(think_times[j])
        row = {
            "case": case_id, "ply": j, "move": uci, "role": role,
            "t_cum": t_cum, "t_spent": float(think_times[j]),
            "t_scaled": scale_time(float(think_times[j])),
            "stm": "w" if board.turn == chess.WHITE else "b",
            "in_check": "1" if board.is_check() else "0",
            "mover": mover, "flag": flag,
            "moves_left": int(mate_n - solver_played),   # noto anche in inferenza
            "rating": float(rating), "mate_n": int(mate_n),
            "board_fen": board.fen(),
        }
        pm = board.piece_map()
        for s, col in enumerate(SQUARE_COLS):
            p = pm.get(s)
            row[col] = p.symbol() if p else "."
        rows.append(row)
    return rows


def rows_to_frame(rows: List[dict]) -> pd.DataFrame:
    """Righe -> DataFrame pronto per TimeGNN (timestamp, ordinamento, indice 0..N-1)."""
    df = pd.DataFrame(rows)
    df["ts"] = BASE_TS + pd.to_timedelta(df["t_cum"], unit="s")
    df = df.sort_values(["case", "ply"], kind="stable").reset_index(drop=True)
    return df


def build_event_log(puzzles: pd.DataFrame, seed: int, chunk: int = 5000) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Puzzle (una riga ciascuno) -> (event log, metadati canonici per puzzle)."""
    rng = np.random.default_rng(seed)
    frames, meta = [], []
    puzzles = puzzles.sort_values("PuzzleId").reset_index(drop=True)
    for start in range(0, len(puzzles), chunk):
        rows = []
        for p in puzzles.iloc[start:start + chunk].itertuples(index=False):
            board, moves, mirrored = canonicalize_puzzle(p.FEN, p.Moves.split())
            times = [simulate_think_time(p.Rating, move_role(j), rng) for j in range(len(moves))]
            rows.extend(sequence_rows(p.PuzzleId, board, moves, times, p.Rating, p.n))
            meta.append({"case": p.PuzzleId, "n": int(p.n), "rating": float(p.Rating),
                         "start_fen": board.fen(), "moves": " ".join(moves),
                         "m0_time": times[0], "mirrored": mirrored,
                         "fen_orig": p.FEN, "moves_orig": p.Moves, "split": p.split})
        frames.append(rows_to_frame(rows))
    events = pd.concat(frames, ignore_index=True)
    events = events.sort_values(["case", "ply"], kind="stable").reset_index(drop=True)
    return events, pd.DataFrame(meta)

In [ ]:
import importlib, chess_tgnn as ct
importlib.reload(ct)

print("mosse nel vocabolario:", len(ct.UCI_UNIVERSE), "| classi (con EOS):", len(ct.CLASSES))

# Controllo: in forma canonica ogni sequenza è legale e termina con il matto
bad, mirrored = 0, 0
for p in puz.sample(min(2000, len(puz)), random_state=0).itertuples():
    board, moves, mir = ct.canonicalize_puzzle(p.FEN, p.Moves.split())
    mirrored += mir
    b = board.copy()
    try:
        for u in moves:
            b.push_uci(u)
        bad += not b.is_checkmate() or b.turn != chess.BLACK   # matto dato dal Bianco
    except ValueError:
        bad += 1
print(f"sequenze non valide: {bad} | specchiate (risolutore Nero): {mirrored}")
assert bad == 0

In [ ]:
# Esempio: un matto in 2 come righe di event log (colonne delle case omesse)
p = puz[puz.n == 2].iloc[0]
board, moves, mir = ct.canonicalize_puzzle(p.FEN, p.Moves.split())
times = [ct.expected_think_time(p.Rating, ct.move_role(j)) for j in range(len(moves))]
demo = ct.rows_to_frame(ct.sequence_rows(p.PuzzleId, board, moves, times, p.Rating, p.n))
print("posizione iniziale (canonica), mosse:", moves, "| specchiato:", mir)
print(board)
demo[["case", "ply", "move", "role", "mover", "flag", "stm", "in_check",
      "moves_left", "t_spent", "t_scaled", "ts"]]

## Step 6 — Costruzione degli event log (train / val / test)

Ogni puzzle diventa 2*n* righe. Si salvano:
- `events_{split}.parquet`: l'event log per TimeGNN; `board_fen` serve in valutazione per sapere quali mosse sono legali;
- `meta.parquet`: una riga per puzzle (posizione canonica, soluzione, orientamento), usata per il rollout.

In [ ]:
t0 = time.time()
events, meta = ct.build_event_log(puz, seed=SEED)
for split in ("train", "val", "test"):
    ids = set(meta.loc[meta.split == split, "case"])
    events[events.case.isin(ids)].reset_index(drop=True).to_parquet(DATA_DIR / f"events_{split}.parquet")
meta.to_parquet(DATA_DIR / "meta.parquet")

print(f"{len(events):,} nodi, {len(meta):,} puzzle in {time.time()-t0:.0f}s  "
      f"| RAM event log: {events.memory_usage(deep=True).sum()/1e9:.2f} GB")
events.groupby("mate_n").agg(nodi=("case", "size"), puzzle=("case", "nunique"))

In [ ]:
# Il tempo simulato: dipende dal rating (è il punto critico per l'ablation)
fig, ax = plt.subplots(figsize=(8, 3.5))
s = events.sample(min(20_000, len(events)), random_state=0)
ax.scatter(s["rating"], s["t_spent"], s=2, alpha=0.3)
ax.set_yscale("log"); ax.set_xlabel("rating"); ax.set_ylabel("secondi (simulati)")
ax.set_title("Tempo di riflessione simulato vs rating")
plt.tight_layout(); plt.show()
del events, s

## Step 7 — Modulo `chess_tgnn`, parte B: grafi TimeGNN

Usiamo `EventLogTransformer` della libreria (lo stesso componente che le recipe usano al loro interno) e applichiamo **ai dati** due correzioni.

**Tempo causale.** La libreria mette sugli archi il "tempo mancante alla fine del case", che dipende dall'ultima mossa: in inferenza non è noto. Lo sostituiamo con il tempo speso sulla mossa di destinazione dell'arco. Nelle varianti senza tempo il valore è 0.

**Self-loop per il modello time-decay.** `TimeAwareGATConv` non li aggiunge, quindi un nodo non riceve le proprie feature. Con due livelli in cascata il nodo *j* dipende solo dal nodo *j−2*, e su un matto in 1 (2 nodi) l'output sarebbe costante. La cella di verifica qui sotto lo mostra. `GATConv` standard (in `gat_basic`) i self-loop li aggiunge già da solo.

**Vocabolario.** Il fit avviene sul training più un "caso vocabolario" fittizio con tutte le 1968 mosse. Così le classi della libreria coincidono con il nostro vocabolario e nessuna mossa di val/test risulta sconosciuta. Il caso non entra negli scaler, perché le sue feature numeriche valgono −1, che la libreria tratta come "mancante".

In [ ]:
%%writefile -a chess_tgnn.py


# =============================================================================
# PARTE B - Dall'event log ai grafi TimeGNN (Step 7 del notebook)
# =============================================================================

CAT_EVENT: List[str] = SQUARE_COLS + ["stm", "in_check", "mover", "flag"]
NUM_EVENT_BASE: List[str] = ["moves_left"]
NUM_EVENT_TIME: List[str] = ["t_scaled"]          # presente solo nelle varianti "con tempo"
NUM_SEQ: List[str] = ["rating", "mate_n"]

# variante -> (mode TimeGNN, usa il tempo?)
VARIANTS: Dict[str, Tuple[str, bool]] = {
    "M1_basic_notime": ("gat_basic", False),
    "M2_basic_time": ("gat_basic", True),
    "M3_decay_notime": ("gat_time_decay", False),
    "M4_decay_time": ("gat_time_decay", True),
}


def feature_columns(use_time: bool) -> dict:
    """Colonne passate a EventLogTransformer. Senza tempo: niente t_scaled."""
    return dict(
        cat_event=list(CAT_EVENT),
        num_event=NUM_EVENT_BASE + (NUM_EVENT_TIME if use_time else []),
        seq_cols=list(NUM_SEQ), cat_seq=[], num_seq=list(NUM_SEQ),
    )


def make_transformer(mode: str, use_time: bool):
    """EventLogTransformer della libreria, configurato per gli scacchi."""
    from timegnn import EventLogTransformer
    return EventLogTransformer(case_col="case", event_col="move", time_col="ts",
                               mode=mode, **feature_columns(use_time))


def vocabulary_case() -> pd.DataFrame:
    """Caso fittizio con tutte le 1968 mosse, usato SOLO nel fit.

    Effetto: le classi del LabelEncoder della libreria = CLASSES (tutte le mosse
    possibili + EOS), senza toccare la libreria. Le feature numeriche valgono -1,
    che la libreria tratta come "mancante" (num_mask=True): non influenzano gli scaler.
    """
    n = len(UCI_UNIVERSE)
    df = pd.DataFrame({
        "case": ["~vocabulary"] * n, "ply": np.arange(n), "move": UCI_UNIVERSE,
        "ts": [BASE_TS] * n, "stm": "w", "in_check": "0", "mover": "P", "flag": "quiet",
        "moves_left": -1, "t_scaled": -1.0, "rating": -1.0, "mate_n": -1,
    })
    for col in SQUARE_COLS:
        df[col] = "."
    return df


def fit_transformer(tr, train_events: pd.DataFrame):
    """Fit degli encoder SOLO sul training (+ caso vocabolario)."""
    cols = ["case", "ply", "move", "ts"] + CAT_EVENT + NUM_EVENT_BASE + NUM_EVENT_TIME + NUM_SEQ
    fit_df = pd.concat([train_events[cols], vocabulary_case()[cols]], ignore_index=True)
    tr.fit(fit_df)
    classes = list(tr._event_encoder.classes_)   # sola lettura, per verifica
    assert classes == CLASSES, "Vocabolario inatteso: controlla vocabulary_case()"
    return tr


def postprocess_graph(g, t_scaled: np.ndarray, mode: str, use_time: bool):
    """Correzioni applicate AI DATI (la libreria resta intatta).

    1) Tempo causale: sull'arco j->j+1 mettiamo il tempo speso sulla mossa j+1
       (noto quando la mossa j+1 e' stata giocata). Sostituisce il "tempo alla
       fine del case" calcolato dalla libreria, che usa informazione futura.
       Senza tempo: tutti zeri (-> nessuna informazione temporale).
    2) Self-loop per il modello time-decay: senza, il nodo j non vede se stesso
       e dopo due livelli dipende solo dal nodo j-2 (verificato nello Step 7).
       GATConv standard (gat_basic) i self-loop li aggiunge gia' da solo.
    """
    import torch
    n = int(g.num_nodes)
    ei = g.edge_index.reshape(2, -1).long()          # 1 nodo -> shape (2, 0)
    assert ei.size(1) == n - 1, "grafo non a catena?"
    if use_time:
        edge_t = torch.tensor(np.asarray(t_scaled[1:n], dtype=np.float32))
    else:
        edge_t = torch.zeros(n - 1)
    if mode == "gat_basic":
        g.edge_index = ei
        g.edge_attr = edge_t.view(-1, 1)
        if "time" in g:
            del g.time
    else:
        loops = torch.arange(n).repeat(2, 1)
        g.edge_index = torch.cat([ei, loops], dim=1)
        g.time = torch.cat([edge_t, torch.zeros(n)])  # tempo 0 -> decay 1 sui self-loop
    return g


def build_graphs(tr, events: pd.DataFrame, mode: str, use_time: bool,
                 chunk_cases: int = 4000) -> Tuple[list, list, List[str]]:
    """Event log -> (grafi PyG, label per nodo, id dei casi) a blocchi.

    La trasformazione a blocchi limita la RAM: la libreria crea array densi
    (one-hot + padding) per tutto il blocco.
    """
    events = events.sort_values(["case", "ply"], kind="stable").reset_index(drop=True)
    case_ids = sorted(events["case"].unique())      # stesso ordine del groupby della libreria
    graphs, labels, order = [], [], []
    for start in range(0, len(case_ids), chunk_cases):
        ids = case_ids[start:start + chunk_cases]
        sub = events[events["case"].isin(set(ids))].reset_index(drop=True)
        out = tr.transform(sub)
        times = sub.groupby("case", sort=True)["t_scaled"].apply(np.asarray)
        assert len(out.event_features) == len(ids)
        for cid, g, y in zip(ids, out.event_features, out.labels):
            graphs.append(postprocess_graph(g, times[cid], mode, use_time))
            labels.append(y)
            order.append(cid)
    return graphs, labels, order


def legal_class_indices(board_fen: str) -> List[int]:
    """Indici di classe delle mosse legali in una posizione (canonica)."""
    return [CLASS_INDEX[m.uci()] for m in chess.Board(board_fen).legal_moves]

In [ ]:
importlib.reload(ct)

# Verifica su pochi puzzle, per entrambi i "mode" (nessun modello viene istanziato qui)
ev_tr = pd.read_parquet(DATA_DIR / "events_train.parquet")
ev_te = pd.read_parquet(DATA_DIR / "events_test.parquet")
few_train = ev_tr[ev_tr.case.isin(ev_tr.case.drop_duplicates().sample(300, random_state=0))]
few_test = ev_te[ev_te.case.isin(sorted(ev_te.case.unique())[:3])]

for mode in ("gat_basic", "gat_time_decay"):
    tr = ct.fit_transformer(ct.make_transformer(mode, use_time=True), few_train)
    graphs, labels, order = ct.build_graphs(tr, few_test, mode, use_time=True)
    g, y, cid = graphs[0], labels[0], order[0]
    seq = few_test[few_test.case == cid].sort_values("ply")
    decoded = [ct.CLASSES[i] for i in g.event_ids.view(-1).tolist()]
    targets = [ct.CLASSES[i] for i in y.view(-1).tolist()]
    assert decoded == seq["move"].tolist(), "nodi non allineati alle mosse!"
    assert targets == seq["move"].tolist()[1:] + ["EOS"], "label non allineate!"
    print(f"\n[{mode}] caso {cid}: {g.num_nodes} nodi, {g.x.shape[1]} feature per nodo")
    print("  mosse (nodi):   ", decoded)
    print("  label (mossa+1):", targets)
    print("  archi:          ", g.edge_index.tolist())
    tvals = g.edge_attr.view(-1) if mode == "gat_basic" else g.time
    print("  tempo sugli archi:", [round(v, 3) for v in tvals.tolist()])
    print("  t_scaled delle mosse:", seq["t_scaled"].round(3).tolist())
del ev_tr, ev_te

**Come leggere l'output.** Le mosse dei nodi e le label devono essere sfasate di uno, con `EOS` in fondo. Il tempo sull'arco *j→j+1* deve coincidere con `t_scaled` della mossa *j+1*.

Nel modello time-decay gli archi `[k, k]` sono i self-loop, con tempo 0 (decadimento 1). La cella seguente dimostra perché servono: gira in un **processo separato**, perché istanzia `TimeAwareGATConv` e il notebook non deve contenere modelli (vedi Step 9).

In [ ]:
import subprocess, textwrap
demo_code = textwrap.dedent('''
    import warnings; warnings.filterwarnings("ignore")
    import torch
    from torch_geometric.data import Data, Batch
    from timegnn.models.gat_time_decay import DualGATTimeAwareModel
    torch.manual_seed(0)
    m = DualGATTimeAwareModel(num_event_features=8, num_embedding_features=10, embedding_dims=8,
            gat_hidden_dim_event=4, gat_hidden_dim_embed=4, gat_hidden_dim_concat=4,
            output_dim=10, num_heads=2, lambda_decay=1.0).eval()
    x, ids = torch.randn(2, 8), torch.tensor([[1], [2]])
    chain = Data(x=x, event_ids=ids, edge_index=torch.tensor([[0], [1]]), time=torch.tensor([0.5]))
    loops = Data(x=x, event_ids=ids, edge_index=torch.tensor([[0, 0, 1], [1, 0, 1]]),
                 time=torch.tensor([0.5, 0.0, 0.0]))
    with torch.no_grad():
        for name, d in (("solo catena", chain), ("con self-loop", loops)):
            out = m(Batch.from_data_list([d]))
            print(f"{name:14s} logit nodo0 = logit nodo1 ?", torch.allclose(out[0], out[1]))
''')
print(subprocess.run([sys.executable, "-c", demo_code], capture_output=True, text=True).stdout)

## Step 8 — Modulo `chess_tgnn`, parte C: risolvere i puzzle (rollout) e baseline

Le accuratezze "mossa per mossa" non bastano: la traccia chiede se il sistema **guida alla soluzione**. Il rollout funziona così:

1. il sistema sceglie la mossa del risolutore **tra le mosse legali** (mascheramento con python-chess);
2. se ha seguito la soluzione, il difensore risponde con la mossa della soluzione Lichess (la miglior difesa);
3. se ha deviato senza dare matto: con Stockfish installato il difensore risponde col motore; senza motore l'esito è `failed_offline`, contato come fallimento ma riportato a parte;

> **Stockfish è fortemente raccomandato** (`brew install stockfish`). In una prova senza motore, i puzzle chiusi come `failed_offline` erano il 46% per n = 2 e il 90% per n = 5. Senza difensore non si sa se la linea alternativa del modello avrebbe portato al matto, quindi il tasso di soluzione risulta sottostimato.
4. **risolto** = matto dato entro *n* mosse. Un matto alternativo valido è accettato, come su Lichess.

Il sottoinsieme di test per il rollout è **deterministico e identico per tutte le varianti e per l'LLM**, così i confronti sono appaiati (McNemar, Step 12). L'esito per puzzle è salvato in un formato comune (`outcomes.csv`) che riuseremo nella parte 2.

Le baseline B0 e B1 non richiedono training e girano subito. B1 risolve per costruzione tutti i matti in 1: va riportata nel report come riferimento, per ricordare che il GNN *riconosce pattern* e non *cerca*.

In [ ]:
%%writefile -a chess_tgnn.py


# =============================================================================
# PARTE C - Risolvere il puzzle: rollout, baseline, esiti (Step 8 del notebook)
# =============================================================================

@dataclass
class PuzzleState:
    """Stato di un puzzle durante il rollout (tutto in forma canonica)."""
    case: str
    n: int
    rating: float
    start_fen: str                 # posizione PRIMA della mossa dell'avversario
    solution: List[str]            # [m0, s1, d1, s2, ..., sn]
    m0_time: float
    mirrored: bool
    board: chess.Board = None
    moves: List[str] = field(default_factory=list)
    times: List[float] = field(default_factory=list)
    solver_played: int = 0
    on_line: bool = True           # finora il risolutore ha seguito la soluzione?
    status: str = "running"        # running|solved|failed|failed_offline|illegal
    predicted: List[str] = field(default_factory=list)
    confs: List[float] = field(default_factory=list)
    first_correct: Optional[bool] = None
    latency_ms: float = 0.0

    @classmethod
    def from_meta(cls, row) -> "PuzzleState":
        s = cls(case=row.case, n=int(row.n), rating=float(row.rating), start_fen=row.start_fen,
                solution=row.moves.split(), m0_time=float(row.m0_time), mirrored=bool(row.mirrored))
        s.board = chess.Board(s.start_fen)
        s.board.push_uci(s.solution[0])           # la mossa dell'avversario e' data
        s.moves, s.times = [s.solution[0]], [s.m0_time]
        return s


def rollout_subset(meta: pd.DataFrame, per_n: int, seed: int) -> pd.DataFrame:
    """Sottoinsieme deterministico (stesso per tutte le varianti -> test appaiati)."""
    parts = [g.sample(min(per_n, len(g)), random_state=seed) for _, g in meta.groupby("n")]
    return pd.concat(parts).sort_values("case").reset_index(drop=True)


def rollout(states: List[PuzzleState], policy_fn: Callable, engine=None,
            engine_limit: float = 0.05) -> List[PuzzleState]:
    """Gioca i puzzle: il policy_fn sceglie le mosse del risolutore, il difensore
    risponde con la linea della soluzione (o con Stockfish se il risolutore
    ha deviato e il motore e' disponibile). Risolto = matto entro n mosse.

    policy_fn(list[PuzzleState]) -> list[(uci, confidenza)] per i soli stati attivi.
    """
    import chess.engine
    max_n = max(s.n for s in states)
    for _ in range(max_n):
        active = [s for s in states if s.status == "running"]
        if not active:
            break
        t0 = time.perf_counter()
        picks = policy_fn(active)
        dt_ms = (time.perf_counter() - t0) * 1000.0 / len(active)
        for s, (uci, conf) in zip(active, picks):
            s.latency_ms += dt_ms
            k = s.solver_played
            expected = s.solution[2 * k + 1] if 2 * k + 1 < len(s.solution) else None
            if k == 0:
                s.first_correct = (uci == expected)
            mv = chess.Move.from_uci(uci)
            if mv not in s.board.legal_moves:
                s.status = "illegal"
                continue
            s.board.push(mv)
            s.moves.append(uci)
            s.times.append(expected_think_time(s.rating, "solver"))
            s.predicted.append(uci)
            s.confs.append(float(conf))
            s.solver_played += 1
            if uci != expected:
                s.on_line = False
            if s.board.is_checkmate():
                s.status = "solved"
                continue
            if s.solver_played >= s.n or s.board.is_game_over():
                s.status = "failed"
                continue
            reply = None
            if s.on_line and 2 * k + 2 < len(s.solution):
                reply = s.solution[2 * k + 2]
            elif engine is not None:
                reply = engine.play(s.board, chess.engine.Limit(time=engine_limit)).move.uci()
            if reply is None:
                s.status = "failed_offline"   # deviazione non valutabile senza motore
                continue
            s.board.push_uci(reply)
            s.moves.append(reply)
            s.times.append(expected_think_time(s.rating, "def"))
    return states


def outcomes_frame(states: List[PuzzleState], system: str, seed: int) -> pd.DataFrame:
    """Esito per puzzle. Stesso formato che useremo per l'LLM (parte 2)."""
    return pd.DataFrame([{
        "system": system, "seed": seed, "case": s.case, "n": s.n, "rating": s.rating,
        "status": s.status, "solved": s.status == "solved",
        "first_move_correct": bool(s.first_correct),
        "pred_line": " ".join(decanonicalize_uci(m, s.mirrored) for m in s.predicted),
        "true_line": " ".join(decanonicalize_uci(m, s.mirrored) for m in s.solution[1::2]),
        "mean_conf": float(np.mean(s.confs)) if s.confs else np.nan,
        "latency_ms": s.latency_ms,
    } for s in states])


def summarize_outcomes(out: pd.DataFrame) -> pd.DataFrame:
    """Tasso di soluzione e accuratezza della prima mossa per n."""
    g = out.groupby("n")
    return pd.DataFrame({
        "puzzles": g.size(),
        "solve_rate": g["solved"].mean(),
        "first_move_acc": g["first_move_correct"].mean(),
        "offline_rate": g["status"].apply(lambda s: (s == "failed_offline").mean()),
    })


# --- Baseline senza apprendimento ---------------------------------------------

def random_policy(seed: int = 0) -> Callable:
    """B0: mossa legale a caso (pavimento)."""
    rng = np.random.default_rng(seed)

    def fn(states):
        out = []
        for s in states:
            legal = [m.uci() for m in s.board.legal_moves]
            out.append((legal[rng.integers(len(legal))], 1.0 / len(legal)))
        return out
    return fn


def greedy_policy(seed: int = 0) -> Callable:
    """B1: euristica a 1 semimossa: matto > scacco > cattura > caso.

    Non e' un modello appreso: serve a mostrare quanto si ottiene con una
    "ricerca" minima (per n=1 risolve tutto per costruzione).
    """
    rng = np.random.default_rng(seed)

    def fn(states):
        out = []
        for s in states:
            legal = list(s.board.legal_moves)
            mates, checks, caps = [], [], []
            for m in legal:
                if s.board.is_capture(m):
                    caps.append(m)
                if s.board.gives_check(m):
                    b = s.board.copy(stack=False)
                    b.push(m)
                    (mates if b.is_checkmate() else checks).append(m)
            pool = mates or checks or caps or legal
            out.append((pool[rng.integers(len(pool))].uci(), 1.0 / len(pool)))
        return out
    return fn

In [ ]:
importlib.reload(ct)
meta = pd.read_parquet(DATA_DIR / "meta.parquet")
subset = ct.rollout_subset(meta[meta.split == "test"], TRAIN_CFG["rollout_per_n"], SEED)
print(f"rollout su {len(subset)} puzzle di test:", subset.groupby("n").size().to_dict())

engine = None
if STOCKFISH_PATH:
    import chess.engine
    engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)

baseline_tables = {}
for name, policy in (("B0_random", ct.random_policy(0)), ("B1_greedy", ct.greedy_policy(0))):
    states = [ct.PuzzleState.from_meta(r) for r in subset.itertuples(index=False)]
    ct.rollout(states, policy, engine=engine)
    out = ct.outcomes_frame(states, name, 0)
    (RUNS_DIR / f"{name}_s0").mkdir(exist_ok=True)
    out.to_csv(RUNS_DIR / f"{name}_s0" / "outcomes.csv", index=False)
    baseline_tables[name] = ct.summarize_outcomes(out)
if engine is not None:
    engine.quit()

pd.concat(baseline_tables, axis=1).round(3)

## Step 9 — Script di training `train_variant.py`

Ogni variante gira in un **processo Python separato**. Il motivo è tecnico e verificato: in PyG ≥ 2.5 `propagate` viene generato a livello di classe. Dopo che un `GATConv` standard è stato istanziato (serve a `gat_basic`), ogni `TimeAwareGATConv` dello stesso processo fallisce con `unexpected keyword 'edge_attr'`. Un processo per variante evita il problema senza toccare la libreria, e rende ogni run indipendente e riproducibile.

Cosa fa lo script:
1. costruisce i grafi (oppure li legge dalla cache `DATA_DIR/cache/`), fittando `EventLogTransformer` **solo sul training**;
2. crea il modello della libreria (`DualGATModel` o `DualGATTimeAwareModel`);
3. allena con `train_epoch` ed `evaluate_epoch` della libreria, con `EarlyStopping` della libreria sulla *validation* loss, e salva i pesi migliori (le recipe restituiscono l'ultima epoca);
4. valuta sul test in due modi:
   - **teacher forcing**: top-1 e top-3 sulle mosse del risolutore, con mascheramento delle mosse legali;
   - **rollout**: tasso di soluzione e accuratezza della prima mossa per *n*;
5. salva `metrics.json`, `outcomes.csv`, `teacher_forced.csv`, `best.pt`, `transformer.pkl` e il log.

La loss della libreria è calcolata su tutti i nodi, quindi anche sulle mosse del difensore e su `EOS`. Per il training va bene, perché è un segnale ausiliario; le metriche che riportiamo riguardano però solo le mosse del risolutore.

In [ ]:
%%writefile train_variant.py
"""Addestra e valuta UNA variante TimeGNN in un processo dedicato.

GENERATO DAL NOTEBOOK (Step 9). Perche' un processo separato: con PyG >= 2.5,
istanziare un GATConv standard (gat_basic) rompe TimeAwareGATConv (gat_time_decay)
nello stesso processo Python. Un processo per variante evita il problema senza
modificare la libreria.

Uso:  python train_variant.py --variant M4_decay_time --seed 0 \
          --data-dir data --out-dir runs/M4_decay_time_s0 --config train_config.json
"""
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # op non supportate da MPS -> CPU

import argparse
import json
import pickle
import random
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch_geometric.data import Batch

from timegnn.data.pyg import CustomDataset, custom_collate_fn
from timegnn.models.training import train_epoch, evaluate_epoch
from timegnn.train.early_stopping import EarlyStopping

import chess_tgnn as ct


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


def pick_device(name: str) -> str:
    if name != "auto":
        return name
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def build_model(mode: str, n_features: int, cfg: dict):
    """Modelli della libreria, importati solo quando servono (uno per processo)."""
    common = dict(
        num_event_features=n_features,
        num_embedding_features=len(ct.CLASSES),
        embedding_dims=cfg["embedding_dims"],
        gat_hidden_dim_event=cfg["gat_hidden_dim_event"],
        gat_hidden_dim_embed=cfg["gat_hidden_dim_embed"],
        gat_hidden_dim_concat=cfg["gat_hidden_dim_concat"],
        output_dim=len(ct.CLASSES),
        num_heads=cfg["num_heads"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        use_batch_norm=cfg["use_batch_norm"],
        activation=cfg["activation"],
    )
    if mode == "gat_basic":
        from timegnn.models.gat_basic import DualGATModel
        return DualGATModel(**common)
    from timegnn.models.gat_time_decay import DualGATTimeAwareModel
    return DualGATTimeAwareModel(lambda_decay=cfg["lambda_decay"], **common)


def load_or_build_graphs(data_dir: Path, mode: str, use_time: bool):
    """Grafi per (mode, tempo): costruiti una volta e messi in cache su disco."""
    cache = data_dir / "cache" / f"{mode}_{'time' if use_time else 'notime'}.pt"
    if cache.exists():
        log(f"cache grafi: {cache.name}")
        return torch.load(cache, weights_only=False)
    cache.parent.mkdir(parents=True, exist_ok=True)
    ev = {s: pd.read_parquet(data_dir / f"events_{s}.parquet") for s in ("train", "val", "test")}
    tr = ct.make_transformer(mode, use_time)
    log("fit EventLogTransformer sul training...")
    ct.fit_transformer(tr, ev["train"])
    bundle = {"transformer": tr}
    for split, df in ev.items():
        t0 = time.time()
        bundle[split] = ct.build_graphs(tr, df, mode, use_time)
        log(f"grafi {split}: {len(bundle[split][0])} in {time.time() - t0:.0f}s")
    torch.save(bundle, cache)
    return bundle


@torch.no_grad()
def teacher_forced_eval(model, graphs, order, events, device, batch_size):
    """Accuratezza sulle mosse del risolutore, dato il contesto corretto.

    Nodi valutati: j pari (0, 2, 4, ...) -> la label e' la mossa del risolutore.
    'masked' = argmax solo tra le mosse legali (come in inferenza).
    """
    model.eval()
    fens = events.groupby("case", sort=True)["board_fen"].apply(list)
    n_of = events.groupby("case", sort=True)["mate_n"].first()
    recs = []
    for start in range(0, len(graphs), batch_size):
        chunk = graphs[start:start + batch_size]
        batch = Batch.from_data_list(chunk).to(device)
        logits = model(batch).float().cpu()
        ptr = batch.ptr.cpu()
        for i, cid in enumerate(order[start:start + batch_size]):
            lg = logits[ptr[i]:ptr[i + 1]]
            ids = chunk[i].event_ids.view(-1).tolist()
            for j in range(0, len(ids) - 1, 2):
                true = ids[j + 1]
                legal = ct.legal_class_indices(fens[cid][j])
                row = lg[j]
                masked = row[legal]
                top = torch.topk(masked, k=min(3, len(legal))).indices.tolist()
                recs.append({"case": cid, "n": int(n_of[cid]), "solver_move": j // 2 + 1,
                             "top1_raw": int(row.argmax()) == true,
                             "top1": legal[top[0]] == true,
                             "top3": true in [legal[t] for t in top]})
    return pd.DataFrame(recs)


def make_model_policy(model, tr, mode, use_time, device):
    """policy_fn per ct.rollout: stessa featurizzazione del training."""
    @torch.no_grad()
    def fn(states):
        rows = []
        for s in states:
            rows += ct.sequence_rows(s.case, __import__("chess").Board(s.start_fen),
                                     s.moves, s.times, s.rating, s.n)
        graphs, _, order = ct.build_graphs(tr, ct.rows_to_frame(rows), mode, use_time)
        batch = Batch.from_data_list(graphs).to(device)
        logits = model(batch).float().cpu()
        last = (batch.ptr[1:] - 1).cpu()
        pos = {cid: i for i, cid in enumerate(order)}
        out = []
        for s in states:
            row = logits[last[pos[s.case]]]
            legal = [m.uci() for m in s.board.legal_moves]
            idx = [ct.CLASS_INDEX[m] for m in legal]
            probs = torch.softmax(row[idx], dim=0)
            k = int(probs.argmax())
            out.append((legal[k], float(probs[k])))
        return out
    return fn


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", required=True, choices=list(ct.VARIANTS))
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--config", required=True)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--stockfish", default="")
    args = ap.parse_args()

    cfg = json.loads(Path(args.config).read_text())
    data_dir, out_dir = Path(args.data_dir), Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    mode, use_time = ct.VARIANTS[args.variant]
    device = pick_device(args.device)
    set_seed(args.seed)
    log(f"variante={args.variant} mode={mode} tempo={use_time} device={device} seed={args.seed}")

    bundle = load_or_build_graphs(data_dir, mode, use_time)
    tr = bundle["transformer"]
    (g_tr, y_tr, _), (g_va, y_va, _), (g_te, y_te, o_te) = bundle["train"], bundle["val"], bundle["test"]

    bs = cfg["batch_size"]
    train_loader = DataLoader(CustomDataset(g_tr, y_tr), batch_size=bs, shuffle=True,
                              collate_fn=custom_collate_fn)
    val_loader = DataLoader(CustomDataset(g_va, y_va), batch_size=bs, shuffle=False,
                            collate_fn=custom_collate_fn)

    model = build_model(mode, g_tr[0].x.shape[1], cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    log(f"parametri: {n_params:,}  feature per nodo: {g_tr[0].x.shape[1]}")
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    crit = nn.CrossEntropyLoss(ignore_index=-1)
    stopper = EarlyStopping(patience=cfg["patience"], delta=cfg["delta"])

    history, epoch_secs = [], []
    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()
        tl, ta = train_epoch(model, train_loader, opt, crit, device)
        vl, va = evaluate_epoch(model, val_loader, crit, device)
        epoch_secs.append(time.time() - t0)
        history.append({"epoch": epoch, "train_loss": tl, "train_acc": ta, "val_loss": vl, "val_acc": va})
        stop = stopper(vl)
        if stopper.best_loss_updated:
            torch.save(model.state_dict(), out_dir / "best.pt")
        log(f"epoca {epoch:3d}  train {tl:.4f}/{ta:.3f}  val {vl:.4f}/{va:.3f}  "
            f"({epoch_secs[-1]:.0f}s){'  *' if stopper.best_loss_updated else ''}")
        # min_epochs: il modello time-decay resta qualche epoca su un plateau
        # (predice sempre EOS) prima di imparare; non va fermato troppo presto.
        if stop and epoch >= cfg.get("min_epochs", 0):
            log("early stopping")
            break
    model.load_state_dict(torch.load(out_dir / "best.pt", map_location=device))

    # --- valutazione 1: teacher forcing sull'intero test ---------------------
    ev_test = pd.read_parquet(data_dir / "events_test.parquet", columns=["case", "ply", "board_fen", "mate_n"])
    ev_test = ev_test.sort_values(["case", "ply"])
    tf = teacher_forced_eval(model, g_te, o_te, ev_test, device, bs)
    tf.to_csv(out_dir / "teacher_forced.csv", index=False)

    # --- valutazione 2: rollout (risoluzione completa) su un sottoinsieme ----
    meta = pd.read_parquet(data_dir / "meta.parquet")
    subset = ct.rollout_subset(meta[meta.split == "test"], cfg["rollout_per_n"], cfg["data_seed"])
    states = [ct.PuzzleState.from_meta(r) for r in subset.itertuples(index=False)]
    engine = None
    if args.stockfish:
        import chess.engine
        engine = chess.engine.SimpleEngine.popen_uci(args.stockfish)
    policy = make_model_policy(model, tr, mode, use_time, device)
    t0 = time.time()
    for start in range(0, len(states), cfg["rollout_batch"]):
        ct.rollout(states[start:start + cfg["rollout_batch"]], policy, engine=engine)
    if engine is not None:
        engine.quit()
    log(f"rollout su {len(states)} puzzle in {time.time() - t0:.0f}s")
    outcomes = ct.outcomes_frame(states, args.variant, args.seed)
    outcomes.to_csv(out_dir / "outcomes.csv", index=False)

    summ = ct.summarize_outcomes(outcomes)
    tf_n = tf.groupby("n")[["top1_raw", "top1", "top3"]].mean()
    metrics = {
        "variant": args.variant, "seed": args.seed, "mode": mode, "use_time": use_time,
        "device": device, "n_params": n_params, "epochs_run": len(history),
        "best_val_loss": stopper.best_loss, "sec_per_epoch": float(np.mean(epoch_secs)),
        "history": history,
        "teacher_forced_by_n": tf_n.reset_index().to_dict(orient="records"),
        "rollout_by_n": summ.reset_index().to_dict(orient="records"),
        "rollout_overall": {"solve_rate": float(outcomes.solved.mean()),
                            "first_move_acc": float(outcomes.first_move_correct.mean())},
    }
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))
    (out_dir / "config.json").write_text(json.dumps({**cfg, **vars(args)}, indent=2))
    with open(out_dir / "transformer.pkl", "wb") as f:
        pickle.dump(tr, f)
    log("fatto: " + json.dumps(metrics["rollout_overall"]))


if __name__ == "__main__":
    main()

## Step 10 — Lanciare le run: helper, prova rapida e scelta del device

`run_variant` lancia lo script, mostra il log in tempo reale e lo salva in `train.log`. Una run già completata (con `metrics.json` presente) viene **saltata**, quindi il notebook si può interrompere e riprendere.

**MPS o CPU?** Sul Mac la GPU (MPS) non è sempre più veloce: i grafi sono piccoli e alcune operazioni di PyG ricadono sulla CPU (`PYTORCH_ENABLE_MPS_FALLBACK=1`). La cella di benchmark allena M4 per un'epoca su entrambi i device e riporta i secondi per epoca. Il device più veloce va poi impostato in `DEVICE`.

La prima run di ogni (mode, tempo) costruisce e salva in cache i grafi. Con la configurazione completa può richiedere alcuni minuti.

In [ ]:
import subprocess

def run_variant(variant, seed, out_dir, config_path=CONFIG_PATH, device=None, force=False):
    out_dir = Path(out_dir)
    if (out_dir / "metrics.json").exists() and not force:
        print(f"✓ {out_dir.name}: già presente, salto")
        return json.loads((out_dir / "metrics.json").read_text())
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "train_variant.py", "--variant", variant, "--seed", str(seed),
           "--data-dir", str(DATA_DIR), "--out-dir", str(out_dir),
           "--config", str(config_path), "--device", device or DEVICE]
    if STOCKFISH_PATH:
        cmd += ["--stockfish", STOCKFISH_PATH]
    env = {**os.environ, "PYTHONPATH": str(PROJECT_DIR), "PYTHONWARNINGS": "ignore"}
    with open(out_dir / "train.log", "w") as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=env, cwd=PROJECT_DIR)
        for line in proc.stdout:
            if "warn" in line.lower():
                continue
            print(line, end=""); logf.write(line)
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"{variant} seed {seed} fallito: vedi {out_dir/'train.log'}")
    return json.loads((out_dir / "metrics.json").read_text())

In [ ]:
# Benchmark del device: 1 epoca di M4 su CPU e (se c'è) su MPS
bench_cfg = {**TRAIN_CFG, "epochs": 1, "rollout_per_n": 5}
bench_path = PROJECT_DIR / "train_config_bench.json"
bench_path.write_text(json.dumps(bench_cfg))

devices = ["cpu"] + (["mps"] if torch.backends.mps.is_available() else [])
bench = {}
for dev in devices:
    m = run_variant("M4_decay_time", 0, RUNS_DIR / f"_bench_{dev}", bench_path, device=dev, force=True)
    bench[dev] = m["sec_per_epoch"]
print("\nsecondi per epoca:", {k: round(v, 1) for k, v in bench.items()})
print("device consigliato:", min(bench, key=bench.get))

In [ ]:
DEVICE = min(bench, key=bench.get)      # oppure fissarlo a mano: "mps" / "cpu"
print("DEVICE =", DEVICE)

## Step 11 — Tutte le run: 4 varianti × seed

In modalità completa: 4 varianti × 3 seed = 12 run. Il tempo per run si stima dal benchmark (secondi per epoca × circa 15–30 epoche, a seconda dell'early stopping); il rollout pesa poco.

Ordine di grandezza: in una prova su CPU con 8.000 puzzle di training un'epoca durava circa 11 s. Con circa 55.000 puzzle ci si aspetta circa 1–1,5 minuti per epoca, e quindi alcune ore per tutte le 12 run. Il benchmark dello Step 10 dà il numero vero sul vostro M5. È consigliabile lasciare il Mac collegato alla corrente e con lo stop disabilitato (`caffeinate -i` nel terminale).

In [ ]:
all_metrics = []
for seed in SEEDS:
    for variant in ct.VARIANTS:
        print(f"\n===== {variant}  seed {seed} =====")
        all_metrics.append(run_variant(variant, seed, RUNS_DIR / f"{variant}_s{seed}"))
print("\ncompletate:", len(all_metrics), "run")

## Step 12 — Risultati: tabelle, grafici, test statistici

1. **Curve di training**: per controllare overfitting ed early stopping.
2. **Teacher forcing per n**: top-1 e top-3 sulle mosse del risolutore, con mascheramento delle mosse legali; media ± deviazione standard sui seed.
3. **Rollout per n**: tasso di soluzione, cioè la metrica principale, confrontabile con l'LLM.
4. **Ablation del tempo con test di McNemar appaiato** (M1 vs M2 e M3 vs M4): stesso seed, stessi puzzle. Conta solo i puzzle discordanti (risolto da una variante e non dall'altra).

In [ ]:
def load_runs(runs_dir):
    rows, outs, hist = [], [], []
    for d in sorted(Path(runs_dir).iterdir()):
        if d.name.startswith("_") or not d.is_dir():
            continue
        if (d / "metrics.json").exists():
            m = json.loads((d / "metrics.json").read_text())
            for r in m["teacher_forced_by_n"]:
                rows.append({"system": m["variant"], "seed": m["seed"], **r})
            for h in m["history"]:
                hist.append({"system": m["variant"], "seed": m["seed"], **h})
        if (d / "outcomes.csv").exists():
            outs.append(pd.read_csv(d / "outcomes.csv"))
    return pd.DataFrame(rows), pd.concat(outs, ignore_index=True), pd.DataFrame(hist)

tf, outcomes, hist = load_runs(RUNS_DIR)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for (sysname, seed), h in hist.groupby(["system", "seed"]):
    axes[0].plot(h.epoch, h.val_loss, label=f"{sysname} s{seed}")
    axes[1].plot(h.epoch, h.val_acc)
axes[0].set_title("validation loss"); axes[1].set_title("validation accuracy (tutti i nodi)")
axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

(tf.groupby(["system", "n"])[["top1_raw", "top1", "top3"]].agg(["mean", "std"])
   .round(3).unstack("n"))

In [ ]:
solve = (outcomes.groupby(["system", "seed", "n"])["solved"].mean()
                 .groupby(["system", "n"]).agg(["mean", "std"]))
solve_table = solve["mean"].unstack("n").round(3)
display(solve_table)

fig, ax = plt.subplots(figsize=(9, 4))
systems = list(solve_table.index)
width = 0.8 / len(systems)
for i, s in enumerate(systems):
    ax.bar(np.array(solve_table.columns) + (i - len(systems) / 2) * width + width / 2,
           solve_table.loc[s], width, label=s,
           yerr=solve["std"].unstack("n").loc[s].fillna(0))
ax.set_xlabel("n (matto in n)"); ax.set_ylabel("tasso di soluzione")
ax.set_title("Rollout sul test: puzzle risolti per n"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
from scipy.stats import binomtest

def mcnemar(a, b):
    # Test esatto di McNemar su esiti appaiati (bool). Ritorna (b, c, p).
    b_ = int((a & ~b).sum()); c_ = int((~a & b).sum())
    p = binomtest(b_, b_ + c_, 0.5).pvalue if b_ + c_ > 0 else 1.0
    return b_, c_, p

pairs = [("M1_basic_notime", "M2_basic_time"), ("M3_decay_notime", "M4_decay_time")]
rows = []
for no_t, with_t in pairs:
    for seed in SEEDS:
        A = outcomes[(outcomes.system == no_t) & (outcomes.seed == seed)].set_index("case")
        B = outcomes[(outcomes.system == with_t) & (outcomes.seed == seed)].set_index("case")
        common = A.index.intersection(B.index)
        for n in ["tutti"] + sorted(A.n.unique()):
            idx = common if n == "tutti" else common[A.loc[common, "n"] == n]
            b_, c_, p = mcnemar(A.loc[idx, "solved"].values, B.loc[idx, "solved"].values)
            rows.append({"confronto": f"{no_t} vs {with_t}", "seed": seed, "n": n,
                         "solo_senza_tempo": b_, "solo_con_tempo": c_,
                         "delta_solve": B.loc[idx, "solved"].mean() - A.loc[idx, "solved"].mean(),
                         "p_value": p})
ablation = pd.DataFrame(rows)
ablation.round(4)

**Come leggere l'ablation.** `delta_solve > 0` significa che la variante con tempo risolve più puzzle. Il `p_value` dice se la differenza è distinguibile dal caso.

Con il tempo simulato ci aspettiamo differenze piccole e non significative. Va riportato così nel report: è un risultato atteso per costruzione (il tempo è derivato dal rating), non un fallimento.

Con pochi puzzle per *n* il test ha poca potenza, e un "non significativo" vuol dire "non conclusivo", non "nessun effetto".

## Step 13 — Prossimi passi

1. **Configurazione completa**: `SMOKE = False` allo Step 2, poi *Run All*. Dati e run vanno in `data/` e `runs/`, separati da quelli di prova.
2. **Taratura di λ** (opzionale): ripetere M4 con `lambda_decay` ∈ {0,5, 3} per mostrare che il risultato non dipende da un valore fortunato.
3. **Timing reale (notebook 02)**: finali di partite Lichess con `%clk` terminate con matto. È il banco di prova vero per la domanda (2).
4. **Held-out di problemi classici (notebook 03)**: stessa funzione di rollout, stesso formato `outcomes.csv`, *n* fino a 10.
5. **Parte 2 (LLM via Ollama)**: stesso sottoinsieme di test (`ct.rollout_subset`), stesso formato di esito, e poi confronto appaiato GNN vs LLM per *n*.